# Investigación Avanzada: Ondas Difraccion Fresnel

Simulación numérica de vanguardia correspondiente a los Problemas Abiertos y Fronteras de Investigación (Nivel Doctorado).

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft2, ifft2, fftshift

def fresnel_propagation(field, dx, dy, z, wavelength):
    Ny, Nx = field.shape
    k = 2 * np.pi / wavelength
    
    # Spatial frequencies
    fx = np.fft.fftfreq(Nx, d=dx)
    fy = np.fft.fftfreq(Ny, d=dy)
    FX, FY = np.meshgrid(fx, fy)
    
    # Transfer function (Fresnel approximation)
    H = np.exp(1j * k * z) * np.exp(-1j * np.pi * wavelength * z * (FX**2 + FY**2))
    
    # Propagation
    U1 = fft2(field)
    U2 = U1 * H
    return ifft2(U2)

def simulate_fresnel_diffraction():
    N = 1024
    L = 0.01  # 10 mm
    dx = L / N
    dy = dx
    wavelength = 632.8e-9  # He-Ne laser
    z = 0.5  # 50 cm propagation
    
    # Create aperture (square hole)
    x = np.linspace(-L/2, L/2, N)
    y = np.linspace(-L/2, L/2, N)
    X, Y = np.meshgrid(x, y)
    
    # Circular aperture
    aperture = np.zeros((N, N))
    radius = 0.001
    aperture[X**2 + Y**2 <= radius**2] = 1.0
    
    # Add a double slit inside
    slit_width = 0.0002
    slit_dist = 0.001
    double_slit = np.zeros((N, N))
    double_slit[(np.abs(X) < slit_dist/2 + slit_width/2) & 
                (np.abs(X) > slit_dist/2 - slit_width/2) & 
                (np.abs(Y) < 0.002)] = 1.0
                
    field_in = double_slit
    
    field_out = fresnel_propagation(field_in, dx, dy, z, wavelength)
    intensity = np.abs(field_out)**2
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
    ax1.imshow(field_in, extent=[-L/2, L/2, -L/2, L/2], cmap='gray')
    ax1.set_title('Aperture (Double Slit)')
    ax1.set_xlabel('x (m)')
    ax1.set_ylabel('y (m)')
    
    ax2.imshow(np.log1p(intensity * 1e6), extent=[-L/2, L/2, -L/2, L/2], cmap='inferno')
    ax2.set_title(f'Diffraction Pattern at z={z}m')
    ax2.set_xlabel('x (m)')
    ax2.set_ylabel('y (m)')
    
    plt.tight_layout()
    plt.savefig('Ondas_Difraccion_Fresnel.png')
    plt.show()

if __name__ == '__main__':
    simulate_fresnel_diffraction()
